# Đề tài 21 – Báo cáo Phân tích Thực nghiệm DevSecOps (E1, E2, E3)
**Người thực hiện:** Hoàng Gia Long (Long - gialong611)
**Đề tài:** Xây dựng đường ống DevSecOps phát hiện cấu hình sai và lỗ hổng trước triển khai

Notebook này tổng hợp và phân tích toàn diện số liệu thực nghiệm nhằm trả lời các câu hỏi nghiên cứu:
- **RQ1 (E1):** Mức độ hiệu quả phát hiện lỗi (Recall, Precision, F1) của các chính sách Security Gate là bao nhiêu?
- **RQ2 (E1):** Chính sách an ninh tạo ra bao nhiêu cảnh báo và có gây chặn nhầm mã sạch (False Block) không?
- **RQ3 (E2):** Chính sách nào cân bằng tối ưu giữa việc chặn triệt để lỗi nguy hiểm và giảm mệt mỏi cảnh báo (Alert Fatigue)?
- **RQ4 (E3):** Thời gian thực thi pipeline (đo lường qua Trung vị p50, Phân vị p90 và số lần đo N) có đáp ứng yêu cầu vận hành và tránh nguy cơ tê liệt pipeline không?

In [ ]:
import sys
if sys.platform == 'win32':
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Tìm thư mục chứa metrics và raw data
cwd = Path.cwd()
if (cwd / 'results' / 'metrics').exists():
    metrics_dir = cwd / 'results' / 'metrics'
    raw_dir = cwd / 'results' / 'raw'
elif (cwd / 'Code' / 'DevSecOps' / 'tests' / 'results' / 'metrics').exists():
    metrics_dir = cwd / 'Code' / 'DevSecOps' / 'tests' / 'results' / 'metrics'
    raw_dir = cwd / 'Code' / 'DevSecOps' / 'tests' / 'results' / 'raw'
else:
    metrics_dir = Path('.').resolve() / 'results' / 'metrics'
    raw_dir = Path('.').resolve() / 'results' / 'raw'

print(f'Thư mục metrics: {metrics_dir}')
print(f'Thư mục raw data: {raw_dir}')

## 1. Thực nghiệm E1 & E2: So sánh hiệu quả chính sách Security Gate (G0, G1, G2)
Đọc dữ liệu tổng hợp từ `e2_policy_comparison.json` gồm các chỉ số thống kê:
- **Recall (Độ nhạy):** Tỷ lệ lỗi gieo cài bị chặn thành công trên tổng số lỗi.
- **Precision (Độ chuẩn xác):** Tỷ lệ quyết định chặn đúng trên tổng số lần quyết định chặn.
- **F1-Score:** Trung bình điều hòa giữa Precision và Recall.
- **False Block Rate:** Tỷ lệ mã sạch vô tội bị chặn oan.

In [ ]:
with open(metrics_dir / 'e2_policy_comparison.json', encoding='utf-8') as f:
    e2_data = json.load(f)

df_policies = pd.DataFrame(e2_data['policies'])
display_cols = ['policy', 'recall', 'precision', 'f1', 'false_block_rate', 'TP', 'FN', 'FP', 'TN']
df_policies[display_cols]

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Recall và F1-Score
x = np.arange(len(df_policies))
width = 0.35

bars1 = ax1.bar(x - width/2, df_policies['recall'] * 100, width, label='Recall (%)', color='#1f77b4')
bars2 = ax1.bar(x + width/2, df_policies['f1'] * 100, width, label='F1-score (%)', color='#ff7f0e')
ax1.set_xticks(x)
ax1.set_xticklabels(df_policies['policy'].str.upper())
ax1.set_ylabel('Tỷ lệ (%)')
ax1.set_title('So sánh Recall và F1-Score (G0, G1, G2)')
ax1.set_ylim(0, 115)
ax1.grid(axis='y', linestyle='--', alpha=0.7)
ax1.legend()

for b in list(bars1) + list(bars2):
    h = b.get_height()
    ax1.annotate(f'{h:.1f}%', xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 3),
                 textcoords='offset points', ha='center', va='bottom', fontsize=9)

# Biểu đồ 2: Ma trận nhầm lẫn (Confusion Matrix Counts)
ax2.bar(x - width*1.5, df_policies['TP'], width, label='TP (Chặn đúng lỗi)', color='#2ca02c')
ax2.bar(x - width*0.5, df_policies['FN'], width, label='FN (Bỏ sót lỗi)', color='#d62728')
ax2.bar(x + width*0.5, df_policies['FP'], width, label='FP (Chặn nhầm sạch)', color='#9467bd')
ax2.bar(x + width*1.5, df_policies['TN'], width, label='TN (Cho qua sạch)', color='#8c564b')
ax2.set_xticks(x)
ax2.set_xticklabels(df_policies['policy'].str.upper())
ax2.set_ylabel('Số lượng ca kiểm thử')
ax2.set_title('Phân bổ Ma trận nhầm lẫn (TP, FN, FP, TN)')
ax2.grid(axis='y', linestyle='--', alpha=0.7)
ax2.legend()

plt.tight_layout()
plt.show()

## 2. Phân tích chi tiết chiến lược phân tầng Gate (G1 vs G2)
- **G1 (Chính sách máy móc theo mức nghiêm trọng - Naive Severity-Based):**
  - Chặn tất cả phát hiện từ MEDIUM trở lên.
  - Đạt Recall 100% trên 28 lỗi gieo, nhưng dễ tạo áp lực lớn (alert fatigue) cho lập trình viên vì chặn cả các lỗi cấu hình thông thường (như thiếu logging S3, thiếu limit tài nguyên K8s).
- **G2 (Chính sách thông minh theo ngữ cảnh - Context-Aware Risk-Based):**
  - Ưu tiên BLOCK tuyệt đối: Secret rò rỉ, CVE trong danh mục KEV (đã bị khai thác thực tế), CVE có điểm EPSS cao (>= 10%), SCA/Image có sẵn bản vá, và các khóa kiểm soát tác động cao trong IaC/K8s/CI.
  - **6 lỗi được hạ xuống mức WARN:** D04 (Path traversal Medium), D06 (MD5 hash Medium), D20 (S3 logging Medium), D22 (K8s limits Medium), D23 (K8s latest tag Medium), D28 (CI mutable action Medium).
  - Việc phân tầng này giúp đội ngũ phát triển không bị gián đoạn tiến độ chuyển giao trong khi vẫn đảm bảo an toàn tuyệt đối trước các lỗ hổng có thể khai thác ngay.

## 3. Thực nghiệm E3: Đánh giá thời gian thực thi pipeline (Mục 5.5: p50, p90, N)
Theo phân công của Long & Duy, thời gian pipeline được đánh giá qua:
- **Số lần đo độc lập (N):** Đảm bảo cỡ mẫu thống kê tin cậy.
- **Trung vị (p50):** Tránh hiện tượng giá trị trung bình bị kéo lệch bởi các tác vụ mạng hoặc runner khởi động nguội.
- **Phân vị 90 (p90):** Đảm bảo 90% các PR kết thúc trong ngưỡng chấp nhận được.
- **Tiêu chí tê liệt pipeline:** Thời gian chạy vượt quá 10 phút (600 giây).

In [ ]:
df_runs = pd.read_csv(raw_dir / 'runs.csv')

def calc_stats(series):
    return pd.Series({
        'N': int(len(series)),
        'Trung vị p50 (s)': series.median(),
        'Phân vị p90 (s)': series.quantile(0.90),
        'Phân vị p95 (s)': series.quantile(0.95),
        'Trung bình (s)': series.mean(),
        'Min (s)': series.min(),
        'Max (s)': series.max(),
        'Độ lệch chuẩn (s)': series.std()
    })

stats_by_type = df_runs.groupby('type')['duration_seconds'].apply(calc_stats).unstack()
stats_overall = calc_stats(df_runs['duration_seconds']).to_frame().T
stats_overall.index = ['overall']
summary_duration = pd.concat([stats_by_type, stats_overall])
summary_duration.round(2)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Boxplot phân phối thời gian chạy
groups = [df_runs[df_runs['type'] == 'clean']['duration_seconds'],
          df_runs[df_runs['type'] == 'defect']['duration_seconds'],
          df_runs['duration_seconds']]
ax1.boxplot(groups, tick_labels=['Clean (N=10)', 'Defect (N=28)', 'Tổng thể (N=38)'], patch_artist=True,
            boxprops=dict(facecolor='#ccebc5', color='#2b5c8f'))
ax1.axhline(600, color='red', linestyle='--', linewidth=1.5, label='Ngưỡng tê liệt (10 phút / 600s)')
ax1.set_ylabel('Thời gian chạy (giây)')
ax1.set_title('Phân phối thời gian pipeline theo loại nhánh')
ax1.set_ylim(100, 650)
ax1.grid(axis='y', linestyle='--', alpha=0.7)
ax1.legend()

# Biểu đồ 2: So sánh p50, p90 với Ngưỡng tê liệt
labels = ['Clean', 'Defect', 'Tổng thể']
p50_vals = [summary_duration.loc['clean', 'Trung vị p50 (s)'],
            summary_duration.loc['defect', 'Trung vị p50 (s)'],
            summary_duration.loc['overall', 'Trung vị p50 (s)']]
p90_vals = [summary_duration.loc['clean', 'Phân vị p90 (s)'],
            summary_duration.loc['defect', 'Phân vị p90 (s)'],
            summary_duration.loc['overall', 'Phân vị p90 (s)']]

x = np.arange(len(labels))
ax2.bar(x - 0.2, p50_vals, 0.4, label='Trung vị (p50)', color='#386cb0')
ax2.bar(x + 0.2, p90_vals, 0.4, label='Phân vị 90 (p90)', color='#fdc086')
ax2.axhline(600, color='red', linestyle='--', linewidth=1.5, label='Ngưỡng tê liệt (600s)')
ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.set_ylabel('Giây')
ax2.set_title('Trung vị p50 và Phân vị p90 so với Ngưỡng tê liệt')
ax2.set_ylim(0, 650)
ax2.grid(axis='y', linestyle='--', alpha=0.7)
ax2.legend()

for i in range(len(labels)):
    ax2.text(i - 0.2, p50_vals[i] + 10, f'{p50_vals[i]:.1f}s', ha='center', fontsize=9)
    ax2.text(i + 0.2, p90_vals[i] + 10, f'{p90_vals[i]:.1f}s', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Kết luận khoa học và Khuyến nghị đề tài

Dựa trên các phân tích định lượng từ bộ dữ liệu thực nghiệm:

1. **Hiệu quả ngăn chặn (RQ1):**
   - Nếu không có gate an ninh (**G0**), toàn bộ 28 lỗ hổng đều trôi qua pipeline ra môi trường triển khai (Recall 0%).
   - Chính sách **G1** chặn 100% lỗi từ MEDIUM trở lên (Recall 100%), nhưng áp dụng máy móc dễ gây tắc nghẽn công việc.
   - Chính sách **G2** chặn có chọn lọc 22 lỗi nguy cơ cao (Recall 78.57%), kiểm soát chặt chẽ 100% các lỗ hổng thuộc danh mục KEV, EPSS cao và secret.

2. **Tỷ lệ chặn nhầm (RQ2):**
   - Trên 10 thay đổi sạch (C01–C10) và 8 mẫu đối chứng (N01–N08), tỷ lệ chặn nhầm (False Block Rate) của cả 3 chính sách đều đạt **0.00%**, chứng minh bộ luật Rego được thiết kế chặt chẽ và không gây nhiễu.

3. **Cân bằng ma sát và rủi ro (RQ3):**
   - **G2 là chính sách tối ưu nhất cho thực tế sản xuất**: Giảm thiểu 6 cảnh báo BLOCK không cấp thiết xuống mức WARN (giảm 21.4% số lần gián đoạn pipeline), giúp lập trình viên tập trung xử lý các rủi ro có khả năng bị tấn công thực tế.

4. **Khả năng vận hành và thời gian thực thi (RQ4 - E3):**
   - Trung vị thời gian thực thi pipeline là **147.5 giây (~2.46 phút)**; phân vị 90 đạt **162.3 giây (~2.71 phút)**.
   - Toàn bộ 38 lần chạy đều nằm dưới xa ngưỡng tê liệt 10 phút (600 giây). Pipeline đạt yêu cầu chuyển giao liên tục (CI/CD) linh hoạt và tin cậy.